In [1]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report

def process_data_pipeline(epochs_path, users_path):
    epochs_df = pd.read_csv(epochs_path)
    users_df = pd.read_csv(users_path)
    df = epochs_df.merge(users_df, on="user_id", how="left")
    df["delta_hr"] = df["mean_hr"] - df["baseline_hr"]
    df["delta_spo2"] = df["mean_spo2"] - df["baseline_spo2"]
    df["delta_br"] = df["mean_br"] - df["baseline_br"]
    return df

def execute_isolation_forest(df):
    feature_cols = [
        "mean_hr", "min_hr", "max_hr",
        "mean_spo2", "min_spo2",
        "mean_br", "hr_br_ratio",
        "delta_hr", "delta_spo2", "delta_br"
    ]
    X = df[feature_cols]
    iso_forest = IsolationForest(
        n_estimators=100,
        contamination=0.10,
        random_state=42,
        n_jobs=-1
    )
    df["anomaly_score"] = iso_forest.fit_predict(X)
    return df, iso_forest

def evaluate_results(df):
    df["true_anomaly"] = df["true_latent_state"].apply(
        lambda x: 1 if x in ["normal", "sleep"] else -1
    )
    return classification_report(df["true_anomaly"], df["anomaly_score"])

epochs_path = r"C:\Users\Akshat - Personal\Visual Studio Code\Sensera\sensera-poc\model_dev_v1\health_epochs.csv"
users_path = r"C:\Users\Akshat - Personal\Visual Studio Code\Sensera\sensera-poc\model_dev_v1\health_users.csv"
df_processed = process_data_pipeline("health_epochs.csv", "health_users.csv")
df_scored, model = execute_isolation_forest(df_processed)
evaluation = evaluate_results(df_scored)
print(evaluation)

              precision    recall  f1-score   support

          -1       1.00      0.50      0.66      4060
           1       0.89      1.00      0.94     16100

    accuracy                           0.90     20160
   macro avg       0.94      0.75      0.80     20160
weighted avg       0.91      0.90      0.88     20160

